# WhisperAudioToText: Hugging Face parity and transcription demo

Verification for [keras-hub PR #2689](https://github.com/keras-team/keras-hub/pull/2689), pinned to commit `71a0cf39386b7d3b05897a5f54883167878f07a8`.

Run all cells on a CPU runtime. This notebook runs the PR's checkpoint verification script separately with the Torch and TensorFlow backends. It compares parameter counts and encoder/decoder hidden states, transcribes the bundled audio clip, and saves/reloads the converted preset.

Scope: `openai/whisper-tiny.en`, one seeded synthetic waveform and one real audio clip. Hidden-state comparisons use identical Hugging Face log-mel features and a single decoder start token. This is a parity check, not a dataset-level accuracy evaluation. Saved outputs below were executed locally on CPU; the same cells can be rerun in Colab.

In [1]:
import importlib.util
import subprocess
import sys

# Colab already includes some dependencies. Pin the model libraries used here.
if importlib.util.find_spec("google.colab") is not None:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "keras==3.15.1", "tensorflow==2.21.0", "tensorflow-text==2.21.1",
        "torch==2.15.0", "transformers==5.22.2", "safetensors==0.8.0",
        "librosa==0.11.0", "kagglehub==1.0.3", "kagglesdk==0.1.38",
        "sentencepiece==0.2.3", "rich", "regex", "absl-py", "packaging",
    ], check=True)
print("Dependencies ready.")

Dependencies ready.


In [2]:
import os
import tempfile
from pathlib import Path

COMMIT = "71a0cf39386b7d3b05897a5f54883167878f07a8"
local_repo = os.environ.get("WHISPER_REVIEW_REPO")
if local_repo:
    repo = Path(local_repo).resolve()
else:
    repo = Path(tempfile.mkdtemp(prefix="whisper-review-")) / "keras-hub"
    subprocess.run(["git", "clone", "-q", "https://github.com/bzantium/keras-hub.git", str(repo)], check=True)
    subprocess.run(["git", "checkout", "-q", COMMIT], cwd=repo, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
assert head == COMMIT, head
print("KerasHub commit:", head)
print("Python:", sys.version.split()[0])
from importlib.metadata import version
for package in ["keras", "torch", "tensorflow", "tensorflow-text", "transformers", "safetensors", "librosa"]:
    print(f"{package}: {version(package)}")

work = Path(tempfile.mkdtemp(prefix="whisper-verification-"))
# The upstream script resolves this bundled audio relative to its working directory.
(work / "keras_hub").symlink_to(repo / "keras_hub", target_is_directory=True)
script = repo / "tools/checkpoint_conversion/convert_whisper_checkpoints.py"

def verify(backend):
    env = dict(os.environ, KERAS_BACKEND=backend, CUDA_VISIBLE_DEVICES="",
               PYTHONPATH=str(repo), PYTHONUNBUFFERED="1")
    result = subprocess.run(
        [sys.executable, str(script), "--preset", "whisper_tiny_en"],
        cwd=work, env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        text=True,
    )
    (work / f"{backend}.log").write_text(result.stdout + result.stderr)
    if result.returncode:
        print(result.stdout)
        print(result.stderr[-12000:])
        result.check_returncode()
    output = result.stdout
    print(output[output.index("=== Parameter count ==="):])
    import re
    counts = re.search(r"Keras backbone params: ([\d,]+)\nHF      model params: ([\d,]+)", output)
    assert counts and counts[1] == counts[2], "Parameter counts differ"
    keras_text = re.search(r"Keras generate\(\): (.+)", output)[1]
    hf_text = re.search(r"HF    generate\(\): (.+)", output)[1]
    assert keras_text == hf_text, "Transcriptions differ"
    assert "Reloaded preset" in output
    print("Parameter count and sample transcription match; preset reload succeeded.")


KerasHub commit: 71a0cf39386b7d3b05897a5f54883167878f07a8
Python: 3.11.15
keras: 3.15.1
torch: 2.15.0+cpu
tensorflow: 2.21.0
tensorflow-text: 2.21.1
transformers: 5.22.2
safetensors: 0.8.0
librosa: 0.11.0


## Torch CPU

The script asserts encoder mean absolute error < `1e-3` and decoder mean absolute error < `5e-3`. Both mean and maximum errors are printed. The wrapper additionally asserts exact parameter-count and sample-transcription equality. Reloading is a smoke check; it does not compare reloaded model outputs.

In [3]:
verify("torch")

=== Parameter count ===
Keras backbone params: 37,760,256
HF      model params: 37,760,256
Diff:                  0

=== Numerics on synthetic audio ===
Encoder abs diff: max=4.637e-05 mean=3.181e-06
Decoder abs diff: max=3.433e-05 mean=7.924e-06
Encoder/decoder mean abs diff within tolerance.

=== generate() on a real audio clip ===
Keras generate(): " Intelligence is what you use when you don't know what to do."
HF    generate(): " Intelligence is what you use when you don't know what to do."

=== save_to_preset + reload roundtrip ===
Saved preset to ./whisper_tiny_en/preset.
Reloaded preset from ./whisper_tiny_en/preset: WhisperAudioToText

Done. Local preset at ./whisper_tiny_en/preset.

Parameter count and sample transcription match; preset reload succeeded.


## TensorFlow CPU

Run the same checkpoint and inputs with a fresh TensorFlow backend process.

In [4]:
verify("tensorflow")

=== Parameter count ===
Keras backbone params: 37,760,256
HF      model params: 37,760,256
Diff:                  0

=== Numerics on synthetic audio ===
Encoder abs diff: max=5.913e-05 mean=9.876e-07
Decoder abs diff: max=4.387e-05 mean=7.352e-06
Encoder/decoder mean abs diff within tolerance.

=== generate() on a real audio clip ===
Keras generate(): " Intelligence is what you use when you don't know what to do."
HF    generate(): " Intelligence is what you use when you don't know what to do."

=== save_to_preset + reload roundtrip ===
Saved preset to ./whisper_tiny_en/preset.
Reloaded preset from ./whisper_tiny_en/preset: WhisperAudioToText

Done. Local preset at ./whisper_tiny_en/preset.

Parameter count and sample transcription match; preset reload succeeded.
